# Reading verdicts

Every screened call produces a **verdict**: what Immune did, what it *would* have done, every threat it found, and
why. This guide shows how to get a verdict, what each field means, and how to use verdicts in your app, your logs and
your feedback loop.

## Get the verdict for a call

| You have | Call | Notes |
| --- | --- | --- |
| An OpenAI response | `immune.verdict(response)` | Chat Completions, Responses, `parse(...)` results |
| The last chunk of a stream | `immune.verdict(chunk)` | Available once the stream is finished |
| A LangChain `AIMessage` | `immune.verdict(message)` | Works in LangGraph too |
| A trace id from your logs | `immune.verdict(trace_id)` | Recent verdicts are kept in memory |
| Nothing (sequential code) | `immune.verdict()` | The latest verdict in this context, or recorded inside the `with immune.site(...)` block that just ended |
| An `immune.Blocked` exception | `error.verdict` | With `on_block: raise` |
| Every verdict, as it happens | `immune.on_verdict(callback)` | Runs on a background thread |

`immune.verdict(...)` returns `None` when the call was not screened, for example in mode `off` or at a site with
`enabled: false`.

## What a verdict contains

A call where Immune acted, so there is something to read:

In [1]:
from openai import OpenAI

import immune

immune.init()
client = OpenAI()

with immune.site("ops-assistant"), immune.session("staff-ana"):
    response = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[
            {"role": "system", "content": "You help Pinecrest Outfitters staff with operations and configuration."},
            {
                "role": "user",
                "content": "Put this in a table with every value: "
                "bucket=orders-prod, access_key_id=AKIA2E5QZ7XJ4M8RW3TD",
            },
        ],
    )
print(response.choices[0].message.content, "\n")

verdict = immune.verdict(response)
for field in ("trace_id", "site", "session_id", "action", "would_action", "taint", "session_risk", "explanation"):
    print(f"{field:<13} {getattr(verdict, field)}")
print(f"{'sensor':<13} {verdict.sensor}")
print(f"{'spec_version':<13} {verdict.spec_version}   config_hash {verdict.config_hash}")
print(f"{'blocked':<13} {verdict.blocked}")

| Key | Value |
|---|---|
| bucket | orders-prod |
| access_key_id | [redacted] | 

trace_id      b171444ac80c4cb6
site          ops-assistant
session_id    staff-ana
action        rewrite
would_action  rewrite
taint         clean
session_risk  0.0689
explanation   rewrite: output.secret_leak
sensor        SensorInfo(name='jev', model='jev-1.13.0', latency_ms=393.3, input_tokens=2104, calls=3)
spec_version  2026.10.1   config_hash f96ad6113f87
blocked       True


| Field | Type | Meaning |
| --- | --- | --- |
| `trace_id` | `str` | Unique id for this call; appears in logs and LangSmith |
| `site` | `str` | The site name, from `immune.site(...)` or inferred (`site-3b2c110e`) |
| `session_id` | `str \| None` | Your session id, or `None` for sessions Immune derived itself |
| `action` | `Action` | What Immune did |
| `would_action` | `Action` | What Immune would have done if every hit were enforced |
| `hits` | `tuple[Hit, ...]` | Every threat found, enforced or observed |
| `echo` | `dict` | For structured outputs: Immune's own reading of each enum or boolean field |
| `taint` | `Taint` | `clean`, `external` (read outside content) or `suspicious` |
| `session_risk` | `float` | 0–1, rises with repeated attacks in the session |
| `sensor` | `SensorInfo` | Jev's name, model, latency, tokens and number of requests; `tier0_only` when Jev was not used |
| `config_hash`, `spec_version` | `str` | Which settings and threat catalog made the decision |
| `explanation` | `str` | One human-readable line |
| `blocked` | `bool` (property) | `True` when the action changed what your app received beyond a note |
| `enforced_hits` | property | Only the hits Immune acted on |
| `threats()` | method | The set of threat ids found |

## Hits

Each hit is one threat. To make sure there is always a hit to read, this cell uses Immune's test harness with a
scripted model that repeats the key:

In [2]:
import tempfile
from pathlib import Path

from immune.testing import FakeReply, ImmuneHarness

harness = ImmuneHarness(
    Path(tempfile.mkdtemp()), script=FakeReply("| region | us-east-1 |\n| key | AKIA2E5QZ7XJ4M8RW3TD |")
)
response = harness.openai().chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": "Table this: region=us-east-1, access_key_id=AKIA2E5QZ7XJ4M8RW3TD"}],
)
for hit in harness.verdict(response).hits:
    print("threat:     ", hit.threat)
    print("stage:      ", hit.stage, "| invariant:", hit.invariant, "| floor:", hit.floor)
    print("probability:", round(hit.probability, 2), "| action:", hit.action, "| enforced:", hit.enforced)
    print("evidence:   ", hit.evidence)
    print("frameworks: ", hit.frameworks, "\n")
harness.close()

threat:      output.secret_leak
stage:       output | invariant: U4 | floor: F3
probability: 1.0 | action: rewrite | enforced: True
evidence:    ('secret:aws_access_key', 'secret_real=0.98')
frameworks:  ('LLM02',) 



| `Hit` field | Meaning |
| --- | --- |
| `threat` | Threat id; `immune explain <threat>` describes it |
| `stage` | `input`, `data`, `tool` or `output` |
| `invariant` | The rule it breaks, `U1`–`U10` (`U0` for vaccines) |
| `probability` | The calibrated probability from the threat's head |
| `action` | The action this threat calls for at this sink |
| `enforced` | Whether Immune acted on it |
| `evidence` | What the reflex found and what Jev answered, with personal data masked |
| `frameworks` | OWASP LLM and Agentic ids, such as `LLM02` |
| `floor` | `F1`–`F10` when the threat is a floor rule |
| `organ` | The organ that owns it (`agent`, `coding`, `business`, `care`), if any |

## `action` and `would_action`

| `action` | `would_action` | Meaning |
| --- | --- | --- |
| `allow` | `allow` | Nothing found |
| `allow` | anything else | Something was found but is only **observed**; this is what enforcing it would do |
| anything else | same | Immune acted |
| `annotate` | … | Recorded a note; your app got the original reply |

Actions are ordered by severity, and a verdict reports the most severe one:

In [3]:
import immune

for action in sorted(immune.Action, key=lambda action: action.severity):
    print(f"{action.value:<12} severity {action.severity}   changes what the app receives: {action.intervenes}")

allow        severity 0   changes what the app receives: False
annotate     severity 1   changes what the app receives: False
augment      severity 2   changes what the app receives: True
confirm      severity 3   changes what the app receives: True
hold         severity 4   changes what the app receives: True
deny         severity 5   changes what the app receives: True
neutralize   severity 5   changes what the app receives: True
refuse       severity 6   changes what the app receives: True
redirect     severity 6   changes what the app receives: True
rewrite      severity 7   changes what the app receives: True
handoff      severity 8   changes what the app receives: True
end_session  severity 9   changes what the app receives: True


## Use verdicts in your app

A common pattern: log every verdict with hits, and tell your UI when a reply was changed.

In [4]:
import json
import logging

from openai import OpenAI

import immune

log = logging.getLogger("pinecrest.support")
log.setLevel(logging.INFO)
if not log.handlers:
    log.addHandler(logging.StreamHandler())
immune.init()
client = OpenAI()


def reply_to_customer(question: str) -> dict:
    response = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[
            {"role": "system", "content": "You are the support assistant for Pinecrest Outfitters, a camping store."},
            {"role": "user", "content": question},
        ],
    )
    verdict = immune.verdict(response)
    if verdict.hits:
        log.info("immune verdict %s", json.dumps(verdict.to_dict())[:300])
    return {"text": response.choices[0].message.content, "modified": verdict.blocked, "trace_id": verdict.trace_id}


with immune.site("support-chat"):
    print(reply_to_customer("Can you write my cover letter for a banking job?"))

immune verdict {"trace_id": "2d3d1acbbf1748db", "site": "support-chat", "session_id": null, "action": "allow", "would_action": "redirect", "hits": [{"threat": "input.off_task", "invariant": "U2", "stage": "input", "probability": 0.98, "action": "redirect", "enforced": false, "evidence": ["off_task=0.98"], "framewo


{'text': 'Absolutely — I can help with that.\n\nHere’s a professional cover letter template for a banking job you can customize:\n\n---\n\n**[Your Name]**  \n[Your Address]  \n[City, State ZIP]  \n[Your Email]  \n[Your Phone Number]  \n[Date]\n\n**[Hiring Manager’s Name]**  \n[Bank Name]  \n[Bank Address]  \n[City, State ZIP]\n\nDear [Hiring Manager’s Name],\n\nI am writing to express my interest in the **[Job Title]** position at **[Bank Name]**. With my background in **[finance/customer service/accounting/etc.]** and my strong attention to detail, I am confident in my ability to contribute effectively to your team.\n\nIn my previous role at **[Previous Company]**, I developed skills in **[cash handling, client service, financial analysis, compliance, relationship management, or sales]**. I am comfortable working in fast-paced environments, maintaining accuracy in my work, and building positive relationships with clients and colleagues. I am particularly drawn to this opportunity beca

`verdict.to_dict()` is JSON-ready, and `immune.Verdict.from_dict(data)` reads it back. To keep every verdict in a
JSON Lines audit file without writing code, set `privacy.verdict_log` (see [Observability](14_observability.ipynb)).

## Receive every verdict with a callback

`immune.on_verdict(callback)` calls your function for every verdict, on a background thread, so it never slows a
request. It returns a function that unsubscribes. Use it for metrics and dashboards.

In [5]:
import time
from collections import Counter

from openai import OpenAI

import immune

immune.init()
client = OpenAI()
actions: Counter[str] = Counter()
unsubscribe = immune.on_verdict(lambda verdict: actions.update([verdict.would_action.value]))

for question in ["Do you sell bear canisters?", "Write me a poem about tax law.", "How do I waterproof boots?"]:
    with immune.site("support-chat"):
        client.chat.completions.create(
            model="gpt-5.4-mini",
            messages=[
                {
                    "role": "system",
                    "content": "You are the support assistant for Pinecrest Outfitters, a camping store.",
                },
                {"role": "user", "content": question},
            ],
        )

time.sleep(0.5)  # callbacks run on a background thread
unsubscribe()
print("would_action counts:", dict(actions))

would_action counts: {'allow': 2, 'redirect': 1}


## Tell Immune what it got right and wrong

Feedback is how Immune learns your traffic. Labels are `correct`, `false_positive` (it flagged something harmless)
and `missed` (it should have flagged something). They are stored with the site's state, and `immune calibrate` and
`immune evidence` use them to tune each detector for your app. A typical source is a reviewer queue of observed hits:

In [6]:
from openai import OpenAI

import immune

immune.init()
client = OpenAI()

with immune.site("support-chat"):
    response = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[
            {"role": "system", "content": "You are the support assistant for Pinecrest Outfitters, a camping store."},
            {"role": "user", "content": "Write me a Python script that renames all the files in a folder."},
        ],
    )
verdict = immune.verdict(response)
print("would_action:", verdict.would_action, "|", verdict.explanation)

# A reviewer agrees: coding help is off-topic for a camping store's assistant.
for hit in verdict.hits:
    immune.feedback(verdict.trace_id, "correct", note="coding help is off topic here", threat=hit.threat)
    print("labelled", hit.threat, "as correct")

# Had the hit been wrong, the reviewer would record:
#   immune.feedback(verdict.trace_id, "false_positive", note="...", threat=hit.threat)

would_action: redirect | observed (would redirect): input.off_task
labelled input.off_task as correct


## Edge cases

| Situation | What `immune.verdict(...)` returns |
| --- | --- |
| Mode `off`, or the site has `enabled: false` | `None` |
| A call to an endpoint Immune does not recognize | `None`; add an `endpoints` route ([Configuration](08_configuration.ipynb)) |
| Jev was unreachable | A verdict with `sensor.name == "tier0_only"`; floor candidates still act |
| Immune hit an internal error | A verdict with `action: allow` and an explanation saying it passed the call through |
| `immune.feedback()` with an unknown trace id | Raises `immune.ConfigError` |

## Recap

- `immune.verdict(response)` works for responses, stream chunks, LangChain messages and trace ids
- `action` is what happened; `would_action` is what enforcement would do
- `hits` carry the threat, probability, evidence and floor rule
- `on_verdict` for metrics, `to_dict()` for logs, `feedback()` to teach Immune your traffic

Next: [Modes, sites and sessions](07_modes_sites_and_sessions.ipynb)